# 8. Optional acceleration and orientation

These options require their own training/export/quality checks. Standard OCR remains the default. Current orientation and speculative paths support the plate/512 profile only. They do not make WATERec a Tiny model or establish native rotation invariance.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "waterec_ocr").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the repository")
sys.path.insert(0, str(ROOT))
from waterec_ocr.waterec_config import load_config
from waterec_ocr.waterec_models import default_checkpoint
CONFIG_PATH = os.environ.get("WATEREC_CONFIG", "configs/dummy_alpr.yaml")
config = load_config(CONFIG_PATH)
checkpoint = default_checkpoint(config)
deployment = config["output_dir"] / "deployment"
examples = sorted((config["dataset_root"] / "val").glob("*.png"))
IMAGE_PATH = examples[0] if examples else config["dataset_root"] / "val/example.png"
def run(*args):
    return subprocess.run([sys.executable, "-m", "waterec_ocr.waterec_cli", *args, "--config", CONFIG_PATH], cwd=ROOT, check=True)
print("Profile:", CONFIG_PATH, "variant:", config["variant"], "patch budget:", config["tokens"])


In [ ]:
RUN_GPU_PREPROCESSING = False
if RUN_GPU_PREPROCESSING:
    run("infer", "--backend", "tensorrt", "--gpu-preprocess", "--image", str(IMAGE_PATH))


GPU preprocessing retains the Pillow bicubic policy, while image-file decoding remains on the CPU. It needs cc and nvcc. Train an orientation head on training-only fit/holdout manifests bound to the finalized encoder. Correcting a rotated image can require a second encoder pass; upright accuracy can change. Never build the orientation split from evaluation images.

In [ ]:
RUN_ORIENTATION = False
FIT_MANIFEST = ROOT / "dataset/orientation_fit.json"
HELD_MANIFEST = ROOT / "dataset/orientation_held.json"
if RUN_ORIENTATION:
    run("train-orientation", "--fit-manifest", str(FIT_MANIFEST), "--held-manifest", str(HELD_MANIFEST))
    run("export-orientation")
    run("build-orientation")
    run("infer", "--backend", "tensorrt", "--orientation", "--image", str(IMAGE_PATH))


The speculative path adds a draft head bound to the exact target checkpoint. Its training uses target-generated responses on training data. Run internal selection/full refit before export. Numerical differences and target/draft behavior require new qualification; do not copy historical latency into a new deployment.

In [ ]:
RUN_SPECULATIVE = False
DRAFT_CHECKPOINT = config["output_dir"] / "speculative/full_fit/draft.pt"
if RUN_SPECULATIVE:
    run("train-speculative")
    print("Use the actual draft checkpoint printed by training; edit DRAFT_CHECKPOINT accordingly.")
    run("export-speculative", "--draft-checkpoint", str(DRAFT_CHECKPOINT))
    run("build-speculative")
    run("infer", "--backend", "tensorrt", "--speculative", "--gpu-preprocess", "--image", str(IMAGE_PATH))


## Measured speculative operating point

The standalone plate/512 TensorRT path was replayed on 4,120 images: validation 2,476/2,693; reviewed unseen 879/984; foreign plates 332/443. Total target plus draft: 27,507,168 parameters. RTX 5090 batch 1 measured **1.026 ms RGB / 2.024 ms encoded and 698 MiB GPU**, five warmups and 256 images × three passes.

All frozen speculative strings were preserved, but only **4,118/4,120** match cached FP16: one validation gain and one unseen loss. Ordinary AR remains the default. These numbers do not establish price-tag, edge-device, rotation or dynamic-batch performance for this path. See [the experiment record](../docs/experiments.md) and [scoped aggregates](../results/reference_metrics.json). A new trained target/draft pair must be requalified.